In [1]:
!pip install streamlit scikit-learn pandas numpy matplotlib joblib -q

import numpy as np
import pandas as pd
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 41.7 MB/s eta 0:00:00


In [2]:
np.random.seed(42)

n = 1500

df = pd.DataFrame({
    "age": np.random.randint(60, 86, size=n),
    "sex": np.random.choice(["Female", "Male"], size=n),
    "race_ethnicity": np.random.choice(
        ["Mexican American", "Other Hispanic", "Non-Hispanic White",
         "Non-Hispanic Black", "Non-Hispanic Asian", "Other"],
        size=n
    ),
    "education": np.random.choice(
        ["Less than high school", "High school", "Some college", "College or above"],
        size=n
    ),
    "bmi": np.round(np.random.normal(28, 5, size=n), 1),
    "diabetes": np.random.choice(["No", "Yes"], size=n, p=[0.72, 0.28]),
    "hypertension": np.random.choice(["No", "Yes"], size=n, p=[0.55, 0.45]),
    "stroke": np.random.choice(["No", "Yes"], size=n, p=[0.91, 0.09]),
    "physically_active": np.random.choice(["Yes", "No"], size=n, p=[0.55, 0.45]),
    "diff_walk_quarter_mile": np.random.choice(["No", "Yes"], size=n, p=[0.68, 0.32]),
    "diff_walk_10_steps": np.random.choice(["No", "Yes"], size=n, p=[0.72, 0.28]),
    "diff_stoop": np.random.choice(["No", "Yes"], size=n, p=[0.62, 0.38]),
    "diff_lift_carry": np.random.choice(["No", "Yes"], size=n, p=[0.70, 0.30]),
    "diff_household_chores": np.random.choice(["No", "Yes"], size=n, p=[0.80, 0.20]),
})

# Create a frailty deficit count.
def yes_to_deficit(x):
    return 1 if x == "Yes" else 0

df["deficit_count"] = (
    (df["physically_active"] == "No").astype(int)
    + df["diff_walk_quarter_mile"].map(yes_to_deficit)
    + df["diff_walk_10_steps"].map(yes_to_deficit)
    + df["diff_stoop"].map(yes_to_deficit)
    + df["diff_lift_carry"].map(yes_to_deficit)
    + df["diff_household_chores"].map(yes_to_deficit)
)

# Add age and chronic disease influence to make the simulated outcome more realistic.
df.loc[df["age"] >= 75, "deficit_count"] += np.random.binomial(1, 0.25, size=(df["age"] >= 75).sum())
df.loc[df["stroke"] == "Yes", "deficit_count"] += np.random.binomial(1, 0.35, size=(df["stroke"] == "Yes").sum())
df.loc[df["diabetes"] == "Yes", "deficit_count"] += np.random.binomial(1, 0.20, size=(df["diabetes"] == "Yes").sum())

# Frailty categories.
# 0 deficits = robust
# 1-2 deficits = pre-frail
# 3 or more deficits = frail
df["frailty_status"] = pd.cut(
    df["deficit_count"],
    bins=[-1, 0, 2, 10],
    labels=["Robust", "Pre-frail", "Frail"]
)

df.head()

,age,sex,race_ethnicity,education,bmi,diabetes,hypertension,stroke,physically_active,diff_walk_quarter_mile,diff_walk_10_steps,diff_stoop,diff_lift_carry,diff_household_chores,deficit_count,frailty_status
0,66,Female,Other,Less than high school,32.8,Yes,Yes,No,No,Yes,No,No,Yes,No,3,Frail
1,79,Female,Mexican American,Less than high school,29.7,No,No,No,No,Yes,No,Yes,No,No,3,Frail
2,74,Female,Non-Hispanic Black,High school,36.2,No,No,No,No,Yes,No,Yes,No,No,3,Frail
3,70,Male,Other,High school,36.8,No,No,No,No,No,Yes,No,Yes,No,3,Frail
4,67,Male,Mexican American,High school,36.8,Yes,Yes,No,Yes,No,Yes,No,Yes,No,2,Pre-frail


In [3]:
df["frailty_status"].value_counts(normalize=True).round(3)

,proportion
frailty_status,
Pre-frail,0.568
Frail,0.362
Robust,0.070


In [4]:
outcome = "frailty_status"

predictors = [
    "age",
    "sex",
    "race_ethnicity",
    "education",
    "bmi",
    "diabetes",
    "hypertension",
    "stroke",
    "physically_active",
    "diff_walk_quarter_mile",
    "diff_walk_10_steps",
    "diff_stoop",
    "diff_lift_carry",
    "diff_household_chores",
]

X = df[predictors].copy()
y = df[outcome].copy()

# One-hot encode categorical predictors.
X_encoded = pd.get_dummies(X, drop_first=False)

feature_columns = X_encoded.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=10,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=["Robust", "Pre-frail", "Frail"]))

Accuracy: 0.869

Classification report:
              precision    recall  f1-score   support

       Frail       0.85      0.82      0.84       136
   Pre-frail       0.89      0.88      0.88       213
      Robust       0.81      1.00      0.90        26

    accuracy                           0.87       375
   macro avg       0.85      0.90      0.87       375
weighted avg       0.87      0.87      0.87       375


Confusion matrix:
[[ 26   0   0]
 [  6 188  19]
 [  0  24 112]]


In [5]:
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

joblib.dump(rf_model, "models/frailty_model.pkl")
joblib.dump(feature_columns, "models/feature_columns.pkl")

df.to_csv("data/cleaned_nhanes_frailty.csv", index=False)

print("Saved files:")
print("models/frailty_model.pkl")
print("models/feature_columns.pkl")
print("data/cleaned_nhanes_frailty.csv")

Saved files:
models/frailty_model.pkl
models/feature_columns.pkl
data/cleaned_nhanes_frailty.csv


In [6]:
from google.colab import files

files.download("models/frailty_model.pkl")
files.download("models/feature_columns.pkl")
files.download("data/cleaned_nhanes_frailty.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>